# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**: Emma Pereira and Pranati Patnam

**ID**: ejp99 and pp444

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [2]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\emmaj\OneDrive\Desktop\4750HW\hw5-emmaj`


In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [4]:
percmass = [0.15, 0.4, 0.05, 0.03, 0.02, 0.05, 0.18, 0.04, 0.02, 0.02, 0.01, 0.03]
comash = [0.08, 0.07, 0.05, 0.1, 0.15, 0.02, 0.02, 1, 1, 1, 1, 0.7]
rec = [0, 0.55, 0.15, 0.1, 0, 0.3, 0.4, 0.6, 0.75, 0.8, 0.5, 0]

c1waste = 100 #mg/d
c2waste = 90
c3waste = 120

#find each component mass from total mass
compmass1 = []
compmass2 = []
compmass3 = []

for x in range(1,12)
    push!(compmass1, c1waste*percmass[x])
    push!(compmass2, c2waste*percmass[x])
    push!(compmass3, c3waste*percmass[x])
end

#find total ash and rec material produced
ash1 = []
ash2 = []
ash3 = []

rec1 = []
rec2 = []
rec3 = []

for x in range(1,12)
    push!(ash1, compmass1[x]*comash[x])
    push!(ash2, compmass2[x]*comash[x])
    push!(ash3, compmass3[x]*comash[x])
    push!(rec1, compmass1[x]*rec[x])
    push!(rec2, compmass2[x]*rec[x])
    push!(rec3, compmass3[x]*rec[x])
end

println("In city 1, total ash is ", sum(ash1)/c1waste, " and recycled material is ", sum(rec1)/c1waste)
println("In city 2, total ash is ", sum(ash2)/c2waste, " and recycled material is ", sum(rec2)/c2waste)
println("In city 3, total ash is ", sum(ash3)/c3waste, " and recycled material is ", sum(rec3)/c3waste)

#recycling ash residuals
ashfrac = sum(ash1)/c1waste
resfrac = sum(rec1)/c1waste
resfr = 1-(sum(rec1)/c1waste)
println("Overall residual MRF ash fraction is ", (resfr*0.16 + resfrac*0.14)/100)



In city 1, total ash is 0.16409999999999997 and recycled material is 0.3775
In city 2, total ash is 0.16410000000000002 and recycled material is 0.3775
In city 3, total ash is 0.1641 and recycled material is 0.37750000000000006
Overall residual MRF ash fraction is 0.0015245000000000002


### Problem 1.1 Write Up

The overall ash and recycled material fractions were found to be 0.1641 and 0.3775, respectively. This was found by first finding the mass of each component, multiplying each component by the ash and recyling percent separately, and then dividing each by the total mass.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

### Problem 1.2 Write Up

The decision variables for this problem are as follows:

W(i,j): Representing the waste transported from city (i) to disposal site (j) in units [Mg/day]

R(k,j): Representing the residual waste (ash) transported from disposal (k) to disposal (j) in units [Mg/day]. This count for the waste between MSF to LF and MSF to WTE.

Y(j): Representing the operational status of facility, binary. Would mean fized costs is only applied of facility is on, or if y=1.

#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

### Problem 1.3 Write Up

<img Src ="IMG_1114.jpg">

#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

### Problem 1.4 Write up

<img Src = "Screenshot 2025-12-03 at 1.22.56 AM.jpeg.png">

#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [7]:
waste_model = Model(HiGHS.Optimizer)

#parameters
LF = 1
MRF = 2
WTE = 3
cities = 1:3
facilities = 1:3

dist = [
    5   30   15;
    15  25   10;
    13  45   20
]

MRFtoLF = 32
MRFtoWTE = 15
WTEtoLF = 18

R = 0.3775
AshFrac = 0.1641

wastegen = [100, 90, 120]
cap = [200, 350, 210]

#initialize decision variables

#waste flow 
@variable(waste_model, waste[cities, facilities] >= 0)
#first index is city 1-3 and second is disposal facilies LF, MRF, and waste

#residual mass
@variable(waste_model, MRFLF >= 0) #MRF TO LF
@variable(waste_model, MRFWTE >= 0) #MRF TO WTE

#binary power on and off for each facility
@variable(waste_model, power[facilities], Bin)
pLF = power[1]
pMRF = power[2]
pWTE = power[3]

Ash = AshFrac*(sum(waste[i,WTE] for i in cities) + MRFWTE)

#objective to minimize total cost
FC = 2000*pLF + 1500*pMRF + 2500*pWTE #facility cost
Tip = 50*(waste[1,1] + waste[2,1] + waste[3,1] + MRFLF) + 7*(waste[1,2] + waste[2,2] + waste[3,2]) + 60*(waste[1,3] + waste[2,3] + waste[3,3] + MRFWTE) #tipping fee
RF = R*40*(waste[1,2] + waste[2,2] + waste[3,2]) #Recycling fee
TC_city = 1.5 * sum(dist[i,j] * waste[i,j] for i in cities, j in facilities) #normal waste
TC_res = 1.5*(MRFtoLF * MRFLF + MRFtoWTE * MRFWTE)

AshWTE = AshFrac * (waste[1,3] + waste[2,3] + waste[3,3] + MRFWTE)
TC_Ash = 1.5 * WTEtoLF * AshWTE

#define objective
@objective(waste_model, Min, FC + Tip + RF + TC_city + TC_res + TC_Ash)

#add constraints
#mass balance of waste flow
for i in cities
    @constraint(waste_model, sum(waste[i,j] for j in facilities) == wastegen[i])
end

#residual mass balance
@constraint(waste_model, MRFLF + MRFWTE == (1-R)*sum(waste[i,MRF] for i in cities))

#Capacities
@constraint(waste_model, sum(waste[i,LF] for i in cities) + MRFLF + Ash <= cap[LF]*pLF)
@constraint(waste_model, sum(waste[i,MRF] for i in cities) <= cap[MRF]*pMRF)
@constraint(waste_model, sum(waste[i,WTE] for i in cities) + MRFWTE <= cap[WTE]*pWTE)

# binary facility constraints using Big M
#Binary facility on or off
BigM = sum(wastegen)
for i in cities
    @constraint(waste_model, waste[i,LF] <= BigM*pLF)
    @constraint(waste_model, waste[i,MRF] <= BigM*pMRF)
    @constraint(waste_model, waste[i,WTE] <= BigM*pWTE)
end

@constraint(waste_model, MRFLF +MRFWTE <= BigM*pMRF)

optimize!(waste_model)

Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 17 rows; 14 cols; 53 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [2e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
17 rows, 14 cols, 53 nonzeros  0s
14 rows, 13 cols, 46 nonzeros  0s
Presolve reductions: rows 14(-3); columns 13(-1); nonzeros 46(-7) 

Solving MIP model with:
   14 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   46 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Objec

In [8]:
#show minimized cost
println("Min Cost is ", @show objective_value(waste_model))

#show waste allocation
@show value.(waste)
@show value.(power)
@show value.(MRFLF)
@show value.(MRFWTE)

objective_value(waste_model) = 26775.747697092953
Min Cost is 26775.747697092953
value.(waste) = 2-dimensional DenseAxisArray{Float64,2,...} with index sets:
    Dimension 1, 1:3
    Dimension 2, 1:3
And data, a 3×3 Matrix{Float64}:
 100.0                0.0   0.0
  -0.0               -0.0  90.0
  78.40531164014833   0.0  41.59468835985167
value.(power) = 1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, 1:3
And data, a 3-element Vector{Float64}:
  1.0
 -0.0
  1.0
value.(MRFLF) = 0.0
value.(MRFWTE) = 0.0


0.0

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

### Problem 1.5 and 1.6 Write Up

<img Src = "Screenshot 2025-12-03 at 7.34.43 PM.jpeg.png">

From the JuMP optimization model, the final minimized cost was found to be about $26775.75. The conditions to meet this value is that the MRF facility will be off and the WTE and LF facility must be on. The diagram above also shows the necessary waste flows from each city as well as the flows between facilities. City 1 will send all of it waste to the landfill, city 2 will send all of its waste to WTE, and city 3 will divide it waste between WTE and the LF.

Personally, I think the minimized cost value is a reasonable amount, however we are surprised to see that MRF is not going to be on in the optimal solution. This may be due to the low recyling rate of the facility and the far distance it has relative to the other cities, which may mean the transportation costs outweigh the benefits of recyling.

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

#### Problem 2.1 Write up

<img Src = "Screenshot 2025-12-04 133609.png">

#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

#### Problem 2.2 Write Up

<img Src = "Screenshot 2025-12-04 133759.png">
<img Src = "Screenshot 2025-12-04 133931.png">
<img Src = "Screenshot 2025-12-04 134021.png">

## References

List any external references consulted, including classmates.

This project was worked on with Pranati Patnam. Chat GPT played a role in assisting with question 1 and 2, specifically with debugging code and implementing constraints.